In [1]:
import numpy as np
import fhrr_driver as fd

In [2]:
M_space = fd.fhrr_space(100000)
I_space = fd.fhrr_space(1000)
E_space = fd.fhrr_space(1000)
A_space = fd.fhrr_space(250)
B_space = fd.fhrr_space(250)
C_space = fd.fhrr_space(250)
D_space = fd.fhrr_space(250)
EM_map = fd.fhrr_map(E_space, M_space)
IM_map = fd.fhrr_map(I_space, M_space)

In [3]:


def combine(A_vec = None, B_vec = None, C_vec = None, D_vec = None):
    global A_space, B_space, C_space, D_space
    if A_vec is None:
        A_vec = A_space.init_zeros_vec()
    if B_vec is None:
        B_vec = B_space.init_zeros_vec()
    if C_vec is None:
        C_vec = C_space.init_zeros_vec()
    if D_vec is None:
        D_vec = D_space.init_zeros_vec()
    I_vec = np.concat([A_vec, B_vec, C_vec, D_vec], axis=0)
    return I_vec

def split(I_vec):
    size = 250
    A_vec = I_vec[0:size, :]
    B_vec = I_vec[size:2*size, :]
    C_vec = I_vec[2*size:3*size, :]
    D_vec = I_vec[3*size:4*size, :]
    return A_vec, B_vec, C_vec, D_vec

In [5]:
A_vecs = [A_space.init_random_vec() for _ in range(10)]
B_vecs = [B_space.init_random_vec() for _ in range(10)]
C_vecs = [C_space.init_random_vec() for _ in range(10)]
D_vecs = [D_space.init_random_vec() for _ in range(10)]
I_vecs = [combine(A_vecs[i], B_vecs[i], C_vecs[i], D_vecs[i]) for i in range(10)]
E_vecs = [E_space.init_random_vec() for _ in range(10)]
EM_vecs = [EM_map.norm_forwards_norm(E_vecs[i]) for i in range(10)]
IM_vecs = [IM_map.norm_forwards_norm(I_vecs[i]) for i in range(10)]
mem_bundle = M_space.bind_and_bundle([[IM_vecs[i], EM_vecs[i]] for i in range(10)])
print(mem_bundle.shape)

(100000, 1)


In [6]:
def recall(Mem_vec, A_vec = None, B_vec = None, C_vec = None, D_vec = None):
    I_vec = combine(A_vec, B_vec, C_vec, D_vec)
    IM_vec = IM_map.forwards_norm(I_vec)
    EM_vec = M_space.norm_unbind(Mem_vec, IM_vec)
    clean_EM_vec = EM_map.norm_forwards_norm(EM_map.norm_backwards_norm(EM_vec))
    complete_IM_vec = M_space.normalize(M_space.norm_unbind(Mem_vec, clean_EM_vec))
    complete_I_vec = IM_map.norm_backwards_norm(complete_IM_vec)
    A_vec2, B_vec2, C_vec2, D_vec2 = split(complete_I_vec)
    return A_vec2, B_vec2, C_vec2, D_vec2

In [15]:
i_1 = 0
i_2 = 2
i_3 = 1
A_vec, B_vec, C_vec, D_vec = recall(mem_bundle, A_vec=A_vecs[i_1], C_vec=C_vecs[i_2])
print(B_space.similarity_R(B_vecs[i_1], B_vec), B_space.similarity_R(A_vecs[i_1], A_vec), B_space.similarity_R(B_vecs[i_2], B_vec), B_space.similarity_R(B_vecs[i_3], B_vec))
for _ in range(10):
    A_vec, B_vec, C_vec, D_vec = recall(mem_bundle, A_vec, B_vec, C_vec, D_vec)
    print(B_space.similarity_R(B_vecs[i_1], B_vec), B_space.similarity_R(A_vecs[i_1], A_vec), B_space.similarity_R(B_vecs[i_2], B_vec), B_space.similarity_R(B_vecs[i_3], B_vec))
    # print(B_space.similarity_C(B_vecs[0], B_vec))
    # print(B_space.similarity_C(A_vecs[0], A_vec))
    # print(B_space.similarity_C(B_vecs[1], B_vec))
    # print(B_space.similarity_C(B_vecs[2], B_vec))

[[0.58889984]] [[0.80796916]] [[0.47108222]] [[0.04967823]]
[[0.6390061]] [[0.71872533]] [[0.49036727]] [[0.05069865]]
[[0.71878078]] [[0.74121214]] [[0.41432157]] [[0.04553691]]
[[0.80118632]] [[0.80007492]] [[0.32263864]] [[0.05434415]]
[[0.89005321]] [[0.88600044]] [[0.18518597]] [[0.05185727]]
[[0.91853352]] [[0.91586888]] [[0.10978494]] [[0.04454932]]
[[0.92446528]] [[0.92697973]] [[0.07898058]] [[0.04097938]]
[[0.92499899]] [[0.92938524]] [[0.06579141]] [[0.03932472]]
[[0.92532841]] [[0.93001686]] [[0.05997075]] [[0.03781221]]
[[0.92649145]] [[0.9314828]] [[0.05768618]] [[0.03838632]]
[[0.92770049]] [[0.93282153]] [[0.05699947]] [[0.03862723]]


In [39]:
A_vecs = [A_space.init_random_vec() for _ in range(10)]
B_vecs = [B_space.init_random_vec() for _ in range(10)]
C_vecs = [C_space.init_random_vec() for _ in range(10)]
D_vecs = [D_space.init_random_vec() for _ in range(10)]
I_vecs = [[combine(A_vecs[i], B_vecs[(i+k)%10], C_vecs[(i+k*2)%10], D_vecs[(i+k*3)%10]) for i in range(10)] for k in range(1)]
I_bundle_vecs = [I_space.normalize(I_space.bundle(*I_vecs[k])) for k in range(1)]
IM_vecs = [[IM_map.norm_forwards_norm(I_vecs[k][i]) for i in range(10)] for k in range(1)]
mem_bundle = M_space.bind_and_bundle([[IM_vecs[k][i], EM_map.norm_forwards_norm(E_space.init_random_vec())] for i in range(10) for k in range(1)])
print(mem_bundle.shape)


(100000, 1)


In [40]:
def recall_I(Mem_vec, I_vec):
    IM_vec = IM_map.forwards_norm(I_vec)
    EM_vec = M_space.norm_unbind(Mem_vec, IM_vec)
    clean_EM_vec = EM_map.norm_forwards_norm(EM_map.norm_backwards_norm(EM_vec))
    complete_IM_vec = M_space.normalize(M_space.norm_unbind(Mem_vec, clean_EM_vec))
    complete_I_vec = IM_map.norm_backwards_norm(complete_IM_vec)
    return complete_I_vec

In [41]:
i_1 = 0
k = 0
i_2 = 1
A_vec, B_vec, C_vec, D_vec = split(I_bundle_vecs[k])
print(B_space.similarity_R(B_vecs[(i_1+k)%10], B_vec), A_space.similarity_R(A_vecs[i_1], A_vec), B_space.similarity_R(B_vecs[(i_2+k)%10], B_vec))
I_vec = I_space.normalize(I_space.bundle(I_bundle_vecs[k], combine(A_vec = A_vecs[i_1])))
A_vec, B_vec, C_vec, D_vec = split(I_vec)
print(B_space.similarity_R(B_vecs[(i_1+k)%10], B_vec), A_space.similarity_R(A_vecs[i_1], A_vec), B_space.similarity_R(B_vecs[(i_2+k)%10], B_vec))
I_vec = recall_I(mem_bundle, I_vec)
A_vec, B_vec, C_vec, D_vec = split(I_vec)
print(B_space.similarity_R(B_vecs[(i_1+k)%10], B_vec), A_space.similarity_R(A_vecs[i_1], A_vec), B_space.similarity_R(B_vecs[(i_2+k)%10], B_vec))
for _ in range(100):
    I_vec = recall_I(mem_bundle, I_vec)
    A_vec, B_vec, C_vec, D_vec = split(I_vec)
    print(B_space.similarity_R(B_vecs[(i_1+k)%10], B_vec), B_space.similarity_R(A_vecs[i_1], A_vec), B_space.similarity_R(B_vecs[(i_2+k)%10], B_vec))

[[0.23126889]] [[0.28514279]] [[0.30255191]]
[[0.23126889]] [[0.7541215]] [[0.30255191]]
[[0.35323657]] [[0.47542807]] [[0.26454459]]
[[0.38103613]] [[0.46035967]] [[0.25144875]]
[[0.3960011]] [[0.4581245]] [[0.24993319]]
[[0.41263205]] [[0.47425351]] [[0.23734169]]
[[0.4438402]] [[0.49460858]] [[0.21716034]]
[[0.46705998]] [[0.49763345]] [[0.21034676]]
[[0.49490128]] [[0.52755139]] [[0.18806959]]
[[0.52074485]] [[0.5613302]] [[0.17088045]]
[[0.55685841]] [[0.60176819]] [[0.14286008]]
[[0.61623212]] [[0.65377006]] [[0.1168051]]
[[0.68867453]] [[0.72603585]] [[0.07690906]]
[[0.78112463]] [[0.77782083]] [[0.04287132]]
[[0.86373712]] [[0.85619212]] [[-0.00657967]]
[[0.91435122]] [[0.91931291]] [[-0.04597129]]
[[0.94209645]] [[0.93020354]] [[-0.07132182]]
[[0.9489513]] [[0.93603391]] [[-0.08331065]]
[[0.94985899]] [[0.93664176]] [[-0.09097189]]
[[0.94988195]] [[0.93699759]] [[-0.09349531]]
[[0.94872991]] [[0.93813989]] [[-0.09362441]]


KeyboardInterrupt: 